<a href="https://colab.research.google.com/github/Asuskf/low-cost-llm-finetuning/blob/main/supervised-finetuning/lab_02_gemma4_sft.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fine-Tuning Gemma with QLoRA (4-bit) in Google Colab
This notebook provides a complete workflow for fine-tuning the Gemma-4-E2B-it large language model (LLM) using Parameter-Efficient Fine-Tuning (PEFT) techniques, specifically QLoRA (Quantized LoRA). The code is fully optimized to run on a Google Colab NVIDIA T4 GPU.

## 1. Mount Google Drive
Mount Google Drive to save and load models persistently during and after the fine-tuning process.

In [21]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Install Dependencies
Install required libraries for Parameter-Efficient Fine-Tuning (PEFT), quantization (`bitsandbytes`), and optimization.

In [5]:
pip install -U peft bitsandbytes  git+https://github.com/huggingface/transformers.git accelerate torchao -q

## 3. Environment Setup
Configure the PyTorch memory allocator to mitigate VRAM fragmentation during training. This is crucial for avoiding Out-of-Memory (OOM) errors on GPUs with limited memory like the T4.

In [1]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 4. Imports & Core Libraries
Import all necessary modules from `transformers`, `peft`, and `datasets`.

In [2]:
import torch
from datasets import Dataset
from transformers import (
    AutoProcessor,
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training
)

## 5. Model Configuration & Tokenizer
Define the target Hugging Face model ID and load its associated tokenizer and processor. We also ensure a padding token is set, which is required for batched training.

In [3]:
# =========================
# 1. MODEL
# =========================
MODEL_ID = "google/gemma-4-E2B-it"

# =========================
# 2. TOKENIZER + PROCESSOR
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
processor = AutoProcessor.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# =========================
# 3. 4-bit quantization (T4 SAFE)
# =========================
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

# =========================
# 4. LOAD MODEL (4-bit)
# =========================
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16
)

model.config.use_cache = False
model.gradient_checkpointing_enable()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

## 7. Pre-Training Inference Test
Let's test the base model's capabilities before fine-tuning using a standard chat structure.

In [6]:
messages = [
    {"role": "system", "content": "Eres un asistente experto en ML."},
    {"role": "user", "content": "¿Qué es overfitting? Define regularización. solo responde"},
]

# =========================
# 4. Chat template
# =========================
text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = processor(text=text, return_tensors="pt")

# mover a GPU correctamente
inputs = {k: v.to("cuda") for k, v in inputs.items()}

input_len = inputs["input_ids"].shape[-1]

# =========================
# 5. GENERATION OPTIMIZADA T4
# =========================
with torch.inference_mode():
    outputs = model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        temperature=0.0,
        use_cache=True,
        pad_token_id=tokenizer.eos_token_id
    )

# =========================
# 6. Decode correcto
# =========================
response = processor.decode(
    outputs[0][input_len:],
    skip_special_tokens=True
)



[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


In [7]:
print(response)

**Overfitting:** Es cuando un modelo de Machine Learning aprende el ruido y los detalles específicos del conjunto de entrenamiento de manera excesiva, lo que resulta en un rendimiento excelente en los datos de entrenamiento,


## Fine tuning
low cost **(4-bit + LoRA)**

*You must restart*

## 8. LoRA Configuration
Configure Low-Rank Adaptation (LoRA) settings. This freezes the base weights and only trains small, inserted matrices, targeting only ~0.37% of the total parameters.

In [5]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules="all-linear"

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


trainable params: 18,960,384 || all params: 5,123,257,888 || trainable%: 0.3701


## 9. Dataset Preparation
Create a synthetic dataset consisting of instruction-response pairs about ML concepts, simulating a larger training corpus.

In [25]:
# =========================
# 6. DATASET
# =========================
data = [
    {
        "instruction": "Explica overfitting con diferencia entre train y test",
        "response": "Overfitting ocurre cuando el error de entrenamiento baja pero el de test sube, indicando mala generalización."
    },
    {
        "instruction": "Define regularización en machine learning",
        "response": "La regularización penaliza modelos complejos para evitar que memoricen ruido y mejorar la generalización."
    },
    {
        "instruction": "¿Cómo detectas overfitting?",
        "response": "Cuando el modelo tiene bajo error en entrenamiento pero alto en validación o test."
    },
    {
        "instruction": "¿Para qué sirve la regularización?",
        "response": "Sirve para reducir la complejidad del modelo y mejorar su capacidad de generalizar a datos nuevos."
    }
] * 100

dataset = Dataset.from_list(data)

## 10. Tokenization Function
Format each example using an instruction/response template, tokenize it, and mask the padding tokens (setting them to `-100`) so PyTorch ignores them during loss calculation.

In [26]:
# =========================
# 7. TOKENIZATION
# =========================
def tokenize(example):
    text = f"""### Instruction:
{example['instruction']}
### Response:
{example['response']}"""

    tokens = tokenizer(
        text,
        truncation=True,
        max_length=128,
        padding="max_length"
    )

    labels = tokens["input_ids"].copy()

    # SOLO ignorar padding (NO destruir learning signal)
    labels = [
        t if t != tokenizer.pad_token_id else -100
        for t in labels
    ]

    tokens["labels"] = labels
    return tokens

dataset = dataset.map(tokenize)

Map:   0%|          | 0/400 [00:00<?, ? examples/s]

## 11. Training Arguments & Execution (T4 Optimized)
Define training hyperparameters tuned specifically to avoid OOM errors on 15GB VRAM (e.g., `batch_size=1`, `gradient_accumulation=8`, and using the `paged_adamw_8bit` optimizer). Then, initialize the Trainer and start fine-tuning.

In [30]:
# =========================
# 8. TRAINING ARGS (T4 OPTIMIZED)
# =========================
training_args = TrainingArguments(
    output_dir="./results",

    per_device_train_batch_size=10,
    gradient_accumulation_steps=8,

    num_train_epochs=10,
    learning_rate=1e-4,

    fp16=True,
    logging_steps=100,

    save_steps=20,
    optim="paged_adamw_8bit",

    report_to="none"
)

In [31]:
# =========================
# 9. TRAINER
# =========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

In [32]:
# =========================
# 10. TRAIN
# =========================
trainer.train()


Step,Training Loss


TrainOutput(global_step=50, training_loss=0.0, metrics={'train_runtime': 697.042, 'train_samples_per_second': 5.739, 'train_steps_per_second': 0.072, 'total_flos': 7286152593408000.0, 'train_loss': 0.0, 'epoch': 10.0})

## 12. Save the Fine-Tuned Model
Save the newly trained LoRA adapters and tokenizer settings to Google Drive for persistent storage. Note: This only saves the lightweight adapters, not the massive base model.

In [33]:
# =========================
# 11. SAVE MODEL
# =========================
new_model = "/content/drive/MyDrive/Flisol/2026/Fine tuning/Model_lab2/gemma-lora-t4"
new_tokenizer = "/content/drive/MyDrive/Flisol/2026/Fine tuning/Model_lab2/gemma-lora-t4"
model.save_pretrained(new_model)
tokenizer.save_pretrained(new_tokenizer)

('/content/drive/MyDrive/Flisol/2026/Fine tuning/Model_lab2/gemma-lora-t4/tokenizer_config.json',
 '/content/drive/MyDrive/Flisol/2026/Fine tuning/Model_lab2/gemma-lora-t4/chat_template.jinja',
 '/content/drive/MyDrive/Flisol/2026/Fine tuning/Model_lab2/gemma-lora-t4/tokenizer.json')